<a href="https://colab.research.google.com/github/vishakha19-hue/vishakha/blob/main/carpriceprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
CodeAlpha Data Science Internship
TASK 3: Car Price Prediction with Machine Learning
----------------------------------------------------
Goal: Predict a used car's selling price based on features like
brand goodwill, horsepower, mileage, engine size, age, fuel type etc.

"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('./outputs', exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(42)

# ---------------------------------------------------------
# STEP 1: Create Dataset (simulated, realistic car pricing logic)
# ---------------------------------------------------------
n = 1000
brands = ['Maruti', 'Hyundai', 'Honda', 'Toyota', 'Ford', 'BMW', 'Audi', 'Mercedes']
brand_goodwill = {'Maruti': 1.0, 'Hyundai': 1.1, 'Honda': 1.2, 'Toyota': 1.3,
                   'Ford': 1.15, 'BMW': 1.8, 'Audi': 1.85, 'Mercedes': 1.9}
fuel_types = ['Petrol', 'Diesel', 'CNG', 'Electric']
transmission_types = ['Manual', 'Automatic']

df = pd.DataFrame({
    'brand': np.random.choice(brands, n),
    'age_years': np.random.randint(0, 15, n),
    'km_driven': np.random.randint(500, 200000, n),
    'horsepower': np.random.randint(60, 400, n),
    'mileage_kmpl': np.round(np.random.uniform(8, 28, n), 1),
    'engine_cc': np.random.randint(800, 4000, n),
    'fuel_type': np.random.choice(fuel_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'transmission': np.random.choice(transmission_types, n, p=[0.65, 0.35]),
    'owner_count': np.random.randint(1, 4, n),
})

# Simulate realistic price using a formula + noise
df['goodwill_factor'] = df['brand'].map(brand_goodwill)
base_price = (
    df['goodwill_factor'] * 400000
    + df['horsepower'] * 800
    + df['engine_cc'] * 60
    - df['age_years'] * 25000
    - (df['km_driven'] / 1000) * 300
    - df['owner_count'] * 15000
    + df['transmission'].map({'Manual': 0, 'Automatic': 60000})
    + df['fuel_type'].map({'Petrol': 0, 'Diesel': 20000, 'CNG': -30000, 'Electric': 150000})
)
noise = np.random.normal(0, 40000, n)
df['selling_price'] = (base_price + noise).clip(lower=50000).round(0)
df.drop(columns=['goodwill_factor'], inplace=True)

print("First 5 rows of dataset:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nPrice statistics:\n", df['selling_price'].describe())

# ---------------------------------------------------------
# STEP 2: EDA
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
sns.histplot(df['selling_price'], kde=True, color='purple')
plt.title("Distribution of Car Selling Prices")
plt.xlabel("Price (INR)")
plt.savefig('./outputs/task3_price_distribution.png', dpi=120, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='brand', y='selling_price')
plt.xticks(rotation=45)
plt.title("Price by Brand")
plt.savefig('./outputs/task3_price_by_brand.png', dpi=120, bbox_inches='tight')
plt.close()

# ---------------------------------------------------------
# STEP 3: Preprocessing pipeline
# ---------------------------------------------------------
numeric_features = ['age_years', 'km_driven', 'horsepower', 'mileage_kmpl', 'engine_cc', 'owner_count']
categorical_features = ['brand', 'fuel_type', 'transmission']

X = df.drop(columns=['selling_price'])
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# ---------------------------------------------------------
# STEP 4: Train Models
# ---------------------------------------------------------
lin_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])
lin_pipeline.fit(X_train, y_train)
lin_pred = lin_pipeline.predict(X_test)

rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

# ---------------------------------------------------------
# STEP 5: Evaluate
# ---------------------------------------------------------
def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n===== {name} =====")
    print(f"R2 Score : {r2:.4f}")
    print(f"MAE      : {mae:,.0f}")
    print(f"RMSE     : {rmse:,.0f}")
    return r2

r2_lin = evaluate("Linear Regression", y_test, lin_pred)
r2_rf = evaluate("Random Forest Regressor", y_test, rf_pred)

# Actual vs Predicted plot (best model)
best_pred = rf_pred if r2_rf >= r2_lin else lin_pred
best_name = "Random Forest" if r2_rf >= r2_lin else "Linear Regression"

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_pred, alpha=0.5, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted Price ({best_name})")
plt.savefig('./outputs/task3_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.close()

# Feature importance (Random Forest)
ohe = rf_pipeline.named_steps['prep'].named_transformers_['cat']
feature_names = numeric_features + list(ohe.get_feature_names_out(categorical_features))
importances = pd.Series(rf_pipeline.named_steps['model'].feature_importances_, index=feature_names).sort_values()

plt.figure(figsize=(9, 8))
importances.tail(15).plot(kind='barh', color='seagreen')
plt.title("Top 15 Feature Importances - Random Forest")
plt.savefig('./outputs/task3_feature_importance.png', dpi=120, bbox_inches='tight')
plt.close()

print(f"\nBest model: {best_name}")
print("All plots saved to outputs folder.")
print("TASK 3 COMPLETE ✅")


First 5 rows of dataset:
    brand  age_years  km_driven  horsepower  mileage_kmpl  engine_cc  \
0    Audi          5     142694         138          25.4       1941   
1  Toyota         14      81059         127          10.6       1331   
2    Ford          0      42286         232          23.8       3013   
3    Audi          8     138309          71          10.5       3490   
4   Honda          0      36562         347          23.9        889   

  fuel_type transmission  owner_count  selling_price  
0    Petrol       Manual            2       732931.0  
1    Diesel       Manual            3       262127.0  
2    Petrol       Manual            3       824547.0  
3    Petrol       Manual            3       721772.0  
4    Diesel       Manual            1       810017.0  

Dataset shape: (1000, 10)

Price statistics:
 count    1.000000e+03
mean     6.942222e+05
std      2.114308e+05
min      2.089690e+05
25%      5.439002e+05
50%      6.877575e+05
75%      8.493522e+05
max      1.